# Goal of this notebook

In this notebook we're trying to make sense of Spark and its capabilities.

In [ ]:
from datetime import datetime, date
from pyspark.sql import Row, SparkSession 
from pyspark import SparkContext, SparkConf
import numpy as np
import os

import pyarrow

## Connect to Spark Cluster

Before running following block of code, you should run `run-cluster.sh` which starts local Spark cluster and Spark Connect server, so that we can connect our `SparkSession` ti the master.

Also, you need to pack some third-party python modules you're going to use for each worker. There are multiple ways to do it and we will go with the easiest: running everything - this notebook and Spark cluster - in single process.

### A note on two different spark modes

#### 1. Spark Local Mode: The Development Sandbox

```python
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("LocalEnvTest") \
    .getOrCreate()
```

**What it is**: A self-contained, monolithic Spark environment. The Driver, Cluster Manager, and Executors all run within a single JVM tied directly to the notebook's current process.

**Why use it here**: This is the ultimate sandbox. It requires zero infrastructure setup or configuration, making it perfect for rapid prototyping, validating PySpark syntax, and testing data transformations on small sample datasets.

**The Catch**: It does not demonstrate Spark's true distributed power. Compute and memory are strictly bottlenecked by the machine running the notebook.

**Python module packaging**: Because this notebook and the cluster will run in the same process, all workers will have the same python interpreter and environment. That means that if you start this `SparkSession` while in active env, you can easily deliver python modules to it.
#### 2. Spark Connect: The Production Blueprint

```python
spark = SparkSession.builder \
    .appName("Local test") \
    .remote("sc://0.0.0.0:15002") \
    .getOrCreate()
```

**What it is**: A decoupled, client-server architecture. The notebook acts as a lightweight client, sending `DataFrame` operations over the network (via `gRPC`) to be executed by an independent, dedicated Spark cluster. The script `./scripts/run-cluster.sh` needs to be run first to setup the cluster.

**Why use it here**: This demonstrates how Spark operates in the real world. It proves that the logic written in the notebook can be shipped off to a distributed cluster (with dedicated Master and Worker nodes) to crunch massive datasets in parallel.

**The Path Forward**: This is the exact architecture pattern used for deployment. To move to production, the Python code remains completely unchanged; you simply point the `.remote()` URL away from your local test cluster and toward your newly deployed, functional Spark cluster.

**Python module packaging**: The python module packaging process is different from the one used in this notebook and it's described in [Python Package Management](https://spark.apache.org/docs/latest/api/python/tutorial/python_packaging.html). Basically, you need to create  [venv-pack](https://jcristharif.com/venv-pack/index.html) which requires for **each worker to have its python interpreter installed**. In this notebook we would need to pack [pyarrow](https://pypi.org/project/pyarrow/), otherwise we could encounter `ModuleNotFoundError: No module named 'pyarrow'`.

In [ ]:
venv_python_path = "./pyspark_venv/bin/python" 

os.environ['PYSPARK_PYTHON'] = venv_python_path
os.environ['PYSPARK_DRIVER_PYTHON'] = venv_python_path

# spark = SparkSession.builder \
#     .appName("chocolate sales") \
#     .remote("sc://0.0.0.0:15002") \
#     .getOrCreate()

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("LocalEnvTest") \
    .getOrCreate()

In [19]:
# test if the cluster receives a job
df = spark.range(100)
df.write.mode("overwrite").parquet("./data/example")

Go to the web UI (http://localhost:[8080 | 4040]) and see if the cluster received the job.

Now we can load some data (this doesn't involves spark; it's just a reference for dataframe interaction).

In [20]:
df = spark.read.csv("data/calendar.csv", header=True, inferSchema=True)

df.printSchema(3)
df.show(3)
df.describe().show(3)

root
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- week: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)

+----------+----+-----+---+----+-----------+
|      date|year|month|day|week|day_of_week|
+----------+----+-----+---+----+-----------+
|2023-01-01|2023|    1|  1|  52|          6|
|2023-01-02|2023|    1|  2|   1|          0|
|2023-01-03|2023|    1|  3|   1|          1|
+----------+----+-----+---+----+-----------+
only showing top 3 rows
+-------+------------------+-----------------+------------------+------------------+-----------------+
|summary|              year|            month|               day|              week|      day_of_week|
+-------+------------------+-----------------+------------------+------------------+-----------------+
|  count|               731|              731|               731|               731|              731|
|   mean|2023.500

## Data extraction and creating a RDD

Extract data from .h5 and store it as `pyspark.sql.DataFrame` and then convert it to `RDD` around which Spark revolves. **The goal with our data** is to extract heterogeneous metadata (strings, integers, booleans, empty arrays) and turn each H5 file into a single row in a table.

In [21]:
from h5extractor import *
from pyspark.sql.types import *
import pandas as pd
import itertools # for slicing a dict
import tables

In [22]:
metadata_db, parquet_payload = process_single_h5("data/h5data.h5")
dict(itertools.islice(metadata_db.items(), 4))

{'measurement_id': 'e9dc6cbc-60fa-41ef-b6ac-d886d2425441',
 'source_file': 'data/h5data.h5',
 'ingested_at': '2026-04-30T07:26:44.162709',
 'BadPixTest_Config_Dacs_Chip_00_BiasLVDS': 128}

### A test

I used this to test `map_type_to_spark_type`, but for a unit test it is too slow. It can be used as a reference though

```python
print("Starting debug loop...")

for key, value in metadata_db.items():
    spark_type = map_type_to_spark_type(value)
    test_schema = StructType([StructField(key, spark_type, True)])
    test_data = [{key: value}]
    
    try:
        df_test = spark.createDataFrame(data=test_data, schema=test_schema)
        df_test.collect()
        
    except Exception as e:
        print(f"\n--- FAILED ON ---")
        print(f"Key:   {key}")
        print(f"Value: {value}")
        print(f"Type:  {type(value)}")
        print(f"Mapped Spark Type: {spark_type}")
        print(f"Error: {e}")
        break

print("Debug loop finished.")
```

### Create new `pyspark.sql.dataframe`

In [23]:
fields = []

for key, value in metadata_db.items():
    spark_type = map_type_to_spark_type(value)
    field = StructField(key, spark_type, True)
    fields.append(field)

spark_schema = StructType(fields)
df_metadata = spark.createDataFrame([metadata_db], schema=spark_schema)

df_metadata.show()

+--------------------+--------------+--------------------+---------------------------------------+------------------------------------------+------------------------------------------+-----------------------------------+----------------------------------+----------------------------------+-----------------------------------+------------------------------------+-------------------------------------+--------------------------------------+----------------------------------+----------------------------------------+----------------------------------+-----------------------------------+----------------------------------------+-------------------------------+----------------------+-----------------------------+--------------------------------------+-----------------------------------------+--------------------------------------+-----------------------------------+-----------------------------------+----------------------------------+---------------------------------+--------------------------

To process terabytes of data in all the h5 files quickly, we will tell Spark to distribute `process_single_h5` function across our cluster of worker nodes using something like this:

```python
# 1. list of all H5 file paths
file_paths = ["file1.h5", "file2.h5", "file3.h5", "... up to n files"]

# 2. distribute the paths across Spark cluster workers
paths_rdd = spark.sparkContext.parallelize(file_paths)

# 3. each file yields exactly one tuple of (metadata, payload)
extracted_rdd = paths_rdd.map(process_single_h5)

# 4. extract just the metadata dictionaries
metadata_rdd = extracted_rdd.map(lambda result: result[0])

# 5. convert that distributed RDD of dictionaries directly into your DataFrame
df = spark.createDataFrame(metadata_rdd, schema=spark_schema)
```

## Elasticsearch

In [24]:
import json

def get_json_byte_size(data_dict):
    """Returns the size of the dictionary if it were serialized to JSON."""
    try:
        # json.dumps converts it to a string, .encode('utf-8') turns it into bytes
        byte_length = len(json.dumps(data_dict).encode('utf-8'))
        
        # Format it nicely
        if byte_length < 1024:
            return f"{byte_length} Bytes"
        elif byte_length < 1024 * 1024:
            return f"{byte_length / 1024:.2f} KB"
        else:
            return f"{byte_length / (1024 * 1024):.2f} MB"
            
    except TypeError as e:
        return f"Serialization Error: {e}"

# Run this inside your test loop or after processing a single file
print("Elasticsearch Payload (Metadata):", get_json_byte_size(metadata_db))
print("Data Lake Payload (Parquet):     ", get_json_byte_size(parquet_payload))

Elasticsearch Payload (Metadata): 62.61 KB
Data Lake Payload (Parquet):      201.31 MB


In [17]:
from pympler import asizeof

def get_ram_usage(data_dict):
    """Returns the deep memory footprint of a Python object."""
    byte_length = asizeof.asizeof(data_dict)
    
    if byte_length < 1024:
        return f"{byte_length} Bytes"
    elif byte_length < 1024 * 1024:
        return f"{byte_length / 1024:.2f} KB"
    else:
        return f"{byte_length / (1024 * 1024):.2f} MB"

print("RAM Usage (Metadata):", get_ram_usage(metadata_db))
print("RAM Usage (Parquet): ", get_ram_usage(parquet_payload))

RAM Usage (Metadata): 159.66 KB
RAM Usage (Parquet):  545.43 MB


In [29]:
from elasticsearch import Elasticsearch, helpers

def send_to_es(partition_iterator):
    elas = Elasticsearch(
        ["http://localhost:9200"],
        api_key="T0MtbjA1MEJkbTBka2xiVTdqdEw6cUFOc05RZVV5QU5IMDNPZUJ6eE5LUQ==" 
    )
    
    # Create a generator function to yield rows lazily
    def generate_actions():
        for row in partition_iterator:
            yield {
                "_index": "chip_calibrations",
                "_source": row.asDict(recursive=True) 
            }
    
    # helpers.bulk streams the generator and chunks it automatically
    # (Default chunk_size is 500, which you can adjust if needed)
    helpers.bulk(elas, generate_actions())

df_metadata.foreachPartition(send_to_es)

26/04/28 10:37:32 ERROR Executor: Exception in task 3.0 in stage 18.0 (TID 63)4]
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 3386, in main
    process()
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 3375, in process
    out_iter = func(split_index, iterator)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/vboxuser/Desktop/spark/lib/python3.12/site-packages/pyspark/core/rdd.py", line 5306, in pipeline_func
    return func(split, prev_func(split, iterator))
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/vboxuser/Desktop/spark/lib/python3.12/site-packages/pyspark/core/rdd.py", line 5306, in pipeline_func
    return func(split, prev_func(split, iterator))
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/vboxuser/Desktop/spark/lib/python3.12/site-packages/pyspark/core/rdd.py", line 5306, in pipeline_func
    return func(split, p

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 3 in stage 18.0 failed 1 times, most recent failure: Lost task 3.0 in stage 18.0 (TID 63) (Ubuntu-AeroDB-Sparks executor driver): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 3386, in main
    process()
  File "/opt/spark/python/lib/pyspark.zip/pyspark/worker.py", line 3375, in process
    out_iter = func(split_index, iterator)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/vboxuser/Desktop/spark/lib/python3.12/site-packages/pyspark/core/rdd.py", line 5306, in pipeline_func
    return func(split, prev_func(split, iterator))
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/vboxuser/Desktop/spark/lib/python3.12/site-packages/pyspark/core/rdd.py", line 5306, in pipeline_func
    return fun

In [ ]:
# df.explain(True) # very usefull before running transformations